In [32]:
# YouTube comments — data preprocessing
from pathlib import Path
import html
import re
import unicodedata

import numpy as np
import pandas as pd
import nltk
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer

DATA_PATH = Path('data/youtube_comments.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/youtube_comments.csv')
REQUIRED_COLUMNS = ['VideoTitle', 'CommentText']
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
lemmatizer = WordNetLemmatizer()

In [33]:
df = pd.read_csv(DATA_PATH, usecols=REQUIRED_COLUMNS)
df.head()

,VideoTitle,CommentText
0,The ONLY 8 YouTube Shorts Niches That Will Mak...,Claim 60 Done-For-You YouTube Shorts 👉https://...
1,The ONLY 8 YouTube Shorts Niches That Will Mak...,I will try these too 😊
2,The ONLY 8 YouTube Shorts Niches That Will Mak...,Career ladder
3,The ONLY 8 YouTube Shorts Niches That Will Mak...,Very helpful
4,The ONLY 8 YouTube Shorts Niches That Will Mak...,Im not proud of doing “brainrot”


## Inspect the raw dataset

In [34]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.isna().sum().to_frame('missing'))
print('Duplicate rows:', int(df.duplicated().sum()))
df['raw_comment_length'] = df['CommentText'].fillna('').astype(str).str.len()
df['raw_token_count'] = df['CommentText'].fillna('').astype(str).str.split().str.len()
display(df[['raw_comment_length', 'raw_token_count']].describe().round(2))

Shape: (17657, 2)
Columns: ['VideoTitle', 'CommentText']


,missing
VideoTitle,0
CommentText,6


Duplicate rows: 695


,raw_comment_length,raw_token_count
count,17657.00,17657.00
mean,79.16,14.31
std,174.44,28.83
min,0.00,0.00
25%,23.00,4.00
50%,44.00,8.00
75%,82.00,16.00
max,7094.00,1573.00


## Remove non-English comments

In [35]:
def is_english_comment(text: str) -> bool:
    text = html.unescape(str(text))
    letters = [char for char in text if char.isalpha()]
    if len(letters) < 2:
        return False
    latin_ratio = sum(char.lower() in 'abcdefghijklmnopqrstuvwxyz' for char in letters) / len(letters)
    if latin_ratio < 0.85:
        return False
    words = re.findall(r'[A-Za-z]+', text.lower())
    common_english_words = sum(word in ENGLISH_STOP_WORDS for word in words)
    return latin_ratio >= 0.95 or common_english_words > 0

df['is_english'] = df['CommentText'].fillna('').map(is_english_comment)
print(df['is_english'].value_counts())
df = df[df['is_english']].copy()
df = df.drop(columns='is_english')
print('Rows after English-language filtering:', len(df))

is_english
True     16174
False     1483
Name: count, dtype: int64
Rows after English-language filtering: 16174


## Normalize, clean, and tokenize comments

In [36]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
WHITESPACE_RE = re.compile(r"\s+")
WORD_RE = re.compile(r"\b[a-z]+\b")

# Preserve negation words
PROTECTED_WORDS = {
    # Negation
    "no",
    "not",
    "never",
    "neither",
    "none",
    "nobody",
    "nothing",
    "nowhere",
    "cannot",
    # Contrast
    "but",
    "however",
    "although",
    "though",
    "whereas",
    "yet",
    # Intensifiers
    "very",
    "too",
    "most",
    "more",
    "less",
    "rather",
    "enough",
}

CUSTOM_STOPWORDS = set(ENGLISH_STOP_WORDS) - PROTECTED_WORDS


def clean_and_tokenize(text: str) -> tuple[str, list[str]]:
    text = html.unescape(str(text))
    text = unicodedata.normalize("NFKC", text).lower()

    text = URL_RE.sub(" ", text)

    # Preserve negation meaning
    text = re.sub(r"\bcan't\b", "can not", text)
    text = re.sub(r"\bwon't\b", "will not", text)
    text = re.sub(r"\bn't\b", " not", text)

    # Remove punctuation without removing word characters
    text = re.sub(r"[^a-z\s]", " ", text)

    text = WHITESPACE_RE.sub(" ", text).strip()

    tokens = [
        token
        for token in WORD_RE.findall(text)
        if token not in CUSTOM_STOPWORDS and len(token) > 1
    ]
    tokens = [lemmatizer.lemmatize(token) for token in tokens]

    return " ".join(tokens), tokens

cleaned = df['CommentText'].map(clean_and_tokenize)
df['clean_comment'] = cleaned.str[0]
df['comment_tokens'] = cleaned.str[1]
df = df[df['clean_comment'].str.len().gt(0)].copy()
df = df.drop_duplicates(subset=['VideoTitle', 'CommentText']).reset_index(drop=True)
df[['CommentText', 'clean_comment', 'comment_tokens']].head(10)

,CommentText,clean_comment,comment_tokens
0,Claim 60 Done-For-You YouTube Shorts 👉https://...,claim 60 youtube shorts,"[claim, 60, youtube, shorts]"
1,I will try these too 😊,try too,"[try, too]"
2,Career ladder,career ladder,"[career, ladder]"
3,Very helpful,very helpful,"[very, helpful]"
4,Im not proud of doing “brainrot”,im not proud doing brainrot,"[im, not, proud, doing, brainrot]"
5,Thank you bro for info❤,thank bro info,"[thank, bro, info]"
6,What about financial documentary,financial documentary,"[financial, documentary]"
7,oh for gods sake get to the point. You are tal...,oh gods sake point talking issue instead just ...,"[oh, gods, sake, point, talking, issue, instea..."
8,If we upload other’s clips in our videos dont ...,upload clips videos dont ask credit famous ppl,"[upload, clips, videos, dont, ask, credit, fam..."
9,great vudei,great vudei,"[great, vudei]"


## Convert comments to TF-IDF features

In [37]:
tfidf = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    preprocessor=None,
    token_pattern=None,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
)
X_tfidf = tfidf.fit_transform(df['clean_comment'])
print('Rows:', X_tfidf.shape[0])
print('Features:', X_tfidf.shape[1])

Rows: 15442
Features: 15771


## Preprocessing output

In [38]:
df[['VideoTitle', 'CommentText', 'clean_comment', 'comment_tokens']].head()
X_tfidf.shape

(15442, 15771)

## Save cleaned data

In [39]:
clean_data = df.drop(columns=['raw_comment_length', 'raw_token_count', 'comment_tokens'], errors='ignore').copy()
clean_data = clean_data[[column for column in clean_data.columns if column != 'clean_comment'] + ['clean_comment']]
OUTPUT_PATH = DATA_PATH.with_name('youtube_comments_cleaned.csv')
clean_data.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'Saved: {OUTPUT_PATH}')
clean_data.head()

Saved: data/youtube_comments_cleaned.csv


,VideoTitle,CommentText,clean_comment
0,The ONLY 8 YouTube Shorts Niches That Will Mak...,Claim 60 Done-For-You YouTube Shorts 👉https://...,claim 60 youtube shorts
1,The ONLY 8 YouTube Shorts Niches That Will Mak...,I will try these too 😊,try too
2,The ONLY 8 YouTube Shorts Niches That Will Mak...,Career ladder,career ladder
3,The ONLY 8 YouTube Shorts Niches That Will Mak...,Very helpful,very helpful
4,The ONLY 8 YouTube Shorts Niches That Will Mak...,Im not proud of doing “brainrot”,im not proud doing brainrot
